# KEN-E || Iterative Strategy Agent

The `Iterative Strategy Agent` is used to create and modify the strategy documents stored in Firestore for each account, including:
- Business Strategy docs, such as `Customer Strategy`
- Channel Strategy docs, such as `Email Strategy`

The `Iterative Strategy Agent` is a sequential agent that triggers two steps:
1. Have the `strategist_agent` create or modify a document.
2. Have the `refinement_loop` edit the document until all requirements are met. The `refinement_loop` consists of two sub-agents:
  1. The `reviewer_agent` checks the document to make sure all requirements have been met.
  2. The `editor_agent` takes feedback from the reviewer_agent to modify the document until all requirements pass the review.

```
+----------------------------------------------+
| iterative_strategy_agent 🔁                  |
| SequentialAgent:                             |
| 1. Create or Modify a document               |
| 2. Refine the document in loop (≤ 3 times)   |
+-----------------------+----------------------+
                        |
     +------------------+--------------------+
     |                                       |
     v                                       v
+-----------------------------+   +-------------------------+
| strategist_agent            |   | refinement_loop 🔁      |
| Create or Modify doc        |   | LoopAgent               |
|                             |   | 1. Review               |
|                             |   | 2. Edit (fix/exit)      |
+-----------------------------+   +-------------------------+


```

## Setup & Authentication 🔑


In [ ]:
# Check if running in a virtual environment
import sys
import os

# Install packages if not already installed
try:
    import google.generativeai
    import google.adk
    print("✅ Packages already installed")
except ImportError:
    print("📦 Installing required packages...")
    !pip install google-adk>=0.1.0
    !pip install google-generativeai>=0.8.0

# Import all necessary libraries
import re
import asyncio
from IPython.display import display, Markdown
import google.generativeai as genai
#from google import genai
from google.adk.agents import Agent, SequentialAgent, LoopAgent, ParallelAgent
from google.adk.tools import google_search, ToolContext, VertexAiSearchTool
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService, Session
from google.genai.types import Content, Part
from getpass import getpass
try:
    import weave
except ImportError:
    !pip install weave

#weave.init(project_name="quickstart_playground")
print("✅ All libraries are ready to go!")

/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_fields.py:198: UserWarning: Field name "config_type" in "SequentialAgent" shadows an attribute in parent "BaseAgent"
  warnings.warn(


✅ Packages already installed


  from pkg_resources import resource_stream, resource_exists



✅ All libraries are ready to go!


In [ ]:
# --- Securely Configure Your API Key ---

# Prompt the user for their API key securely
api_key = getpass('Enter your Google API Key: ')

# Get Your API Key HERE 👉 https://codelabs.developers.google.com/onramp/instructions#0
# Configure the generative AI library with the provided key
genai.configure(api_key=api_key)

# Set the API key as an environment variable for ADK to use
os.environ['GOOGLE_API_KEY'] = api_key

#print("✅ API Key configured successfully! Let the fun begin.")

Enter your Google API Key: ··········


## Set Test Parameters

In [ ]:
###
# Determine if you will test creating a new doc or modifying an existing one
# Set to 'create' to have KEN-E research Intellipure and create a new business strategy document
# Set to 'modify' to have KEN-E update an existing business strategy document to include a note that Intellipure was established in January 2017

test_action = "create"
test_document = "competitive strategy" # 'business strategy', 'competitive strategy'

## Create Function for Running the Iterative Strategy Agent
The iterative_strategy_agent is called with the following variables:
- query: A prompt that gives the primary instruction to create or modify a document.
- strategy_doc: The strategy document from Firestore that should be modified (if it exists)
- best_practices: The proper best practices document for creating this strategy doc
- reviewer_guidelines: The instructions that the reviewer must follow to ensure this document is complete
- new_information: The new information that should be considered when creating or reviewing the strategy document.

In [ ]:
# --- A Helper Function to Run The Strategy Agent ---

async def run_strategy_agent(
    agent: Agent,
    query: str,
    session: Session,
    user_id: str,
    strategy_doc: str = None,
    best_practices: str = None,
    reviewer_guidelines: str = None,
    new_information: str = None,
    is_router: bool = False
    ):
    """Initializes a runner and executes a query for a given agent and session."""
    print(f"\n🚀 Running query for agent: '{agent.name}' in session: '{session.id}'...")

    # DEBUG: Check strategy_doc value
    print(f"🔍 DEBUG: strategy_doc type: {type(strategy_doc)}")
    print(f"🔍 DEBUG: strategy_doc is None: {strategy_doc is None}")
    if strategy_doc:
        print(f"🔍 DEBUG: strategy_doc length: {len(strategy_doc)} characters")
        print(f"🔍 DEBUG: strategy_doc preview: {strategy_doc[:100]}...")
    else:
        print(f"🔍 DEBUG: strategy_doc is empty or None")

    # Handle different agent types - SequentialAgent and LoopAgent don't have 'tools' attribute
    if hasattr(agent, 'tools'):
        tool_names = [tool.__name__ if hasattr(tool, '__name__') else str(tool) for tool in agent.tools]
        print(f"🔧 Agent tools: {', '.join(tool_names)}")
    else:
        print(f"🔧 Agent type: {type(agent).__name__} (composite agent)")

    runner = Runner(
        agent=agent,
        session_service=session_service,
        app_name=agent.name
    )

    # Construct the message parts
    message_parts = [Part(text=query)]

    # For iterative_strategy_agent (SequentialAgent), we need to pass ALL information
    # so that sub-agents can access what they need
    is_iterative_agent = agent.name == 'iterative_strategy_agent'

    # Pass strategy_doc information
    if strategy_doc is not None and (agent.name == 'strategist_agent' or is_iterative_agent):
        message_parts.append(Part(text=f"THE EXISTING STRATEGY DOCUMENT FOR YOU TO MODIFY: {strategy_doc}"))
        print(f"📄 Provided existing strategy document ({len(strategy_doc)} chars)")
    elif strategy_doc is None and (agent.name == 'strategist_agent' or is_iterative_agent):
        message_parts.append(Part(text=f"No strategy document has been created yet. Your task is to create a new one."))
        print(f"📝 Creating new document")

    # Pass best_practices to strategist_agent and editor_agent
    if best_practices is not None and (agent.name in ['strategist_agent', 'reviewer_agent', 'editor_agent'] or is_iterative_agent):
        message_parts.append(Part(text=f"BEST PRACTICES: {best_practices}"))
        print(f"📋 Provided best_practices")

    # Pass reviewer_guidelines to reviewer_agent
    if reviewer_guidelines is not None and (agent.name == 'reviewer_agent' or is_iterative_agent):
        message_parts.append(Part(text=f"REVIEWER GUIDELINES: {reviewer_guidelines}"))
        print(f"✅ Provided reviewer_guidelines")

    # Pass new_information to strategist_agent
    if new_information is not None and (agent.name == 'strategist_agent' or is_iterative_agent):
        message_parts.append(Part(text=f"NEW INFORMATION: {new_information}"))
        print(f"📊 Provided new_information")

    final_response = ""
    tool_calls_count = 0
    total_prompt_tokens = 0
    total_response_tokens = 0

    # Track per-agent token usage
    agent_token_usage = {}

    try:
        async for event in runner.run_async(
            user_id=user_id,
            session_id=session.id,
            new_message=Content(parts=message_parts, role="user")
        ):
            # Track token usage per agent
            if hasattr(event, 'usage_metadata') and event.usage_metadata:
                # Get the current agent name from the event
                current_agent = getattr(event, 'author', 'unknown')

                # Initialize agent entry if not exists
                if current_agent not in agent_token_usage:
                    agent_token_usage[current_agent] = {
                        'prompt_tokens': 0,
                        'response_tokens': 0,
                        'model': 'unknown'
                    }

                # Add token counts
                if hasattr(event.usage_metadata, 'prompt_token_count'):
                    tokens = event.usage_metadata.prompt_token_count or 0
                    agent_token_usage[current_agent]['prompt_tokens'] += tokens
                    total_prompt_tokens += tokens

                if hasattr(event.usage_metadata, 'candidates_token_count'):
                    tokens = event.usage_metadata.candidates_token_count or 0
                    agent_token_usage[current_agent]['response_tokens'] += tokens
                    total_response_tokens += tokens

                # Try to get model info from agent definitions
                if current_agent == 'strategist_agent':
                    agent_token_usage[current_agent]['model'] = 'gemini-1.5-pro-002'
                elif current_agent == 'reviewer_agent':
                    agent_token_usage[current_agent]['model'] = 'gemini-1.5-flash-latest'
                elif current_agent == 'editor_agent':
                    agent_token_usage[current_agent]['model'] = 'gemini-1.5-flash-latest'

            if not is_router:
                # Simplified logging - only show important events
                if hasattr(event, 'tool_call') and event.tool_call:
                    tool_calls_count += 1
                    tool_name = event.tool_call.name if hasattr(event.tool_call, 'name') else 'unknown'
                    current_agent = getattr(event, 'author', 'unknown')
                    print(f"🔍 Tool call #{tool_calls_count}: {tool_name} (by {current_agent})")
                elif hasattr(event, 'content') and event.content:
                    # Check for function calls in a cleaner way
                    if hasattr(event.content, 'parts') and event.content.parts:
                        for part in event.content.parts:
                            if hasattr(part, 'function_call') and part.function_call:
                                func_name = part.function_call.name if hasattr(part.function_call, 'name') else 'unknown'
                                current_agent = getattr(event, 'author', 'unknown')
                                print(f"🔧 Function call: {func_name} (by {current_agent})")

            if event.is_final_response():
                # Add null checks to prevent NoneType error
                if event.content and hasattr(event.content, 'parts') and event.content.parts and len(event.content.parts) > 0:
                    # Handle text parts only, skip function calls
                    text_parts = []
                    for part in event.content.parts:
                        if hasattr(part, 'text') and part.text:
                            text_parts.append(part.text)
                    final_response = ' '.join(text_parts) if text_parts else "No text response available"
                else:
                    final_response = "No response content available"
    except Exception as e:
        final_response = f"An error occurred: {e}"
        print(f"❌ Error: {e}")

    if not is_router:
        total_tokens = total_prompt_tokens + total_response_tokens

        print(f"\n📊 Token Usage Summary:")
        print(f"   • Total tool calls: {tool_calls_count}")

        # Display per-agent breakdown
        if agent_token_usage:
            print(f"\n   Per-Agent Breakdown:")
            for agent_name, usage in agent_token_usage.items():
                agent_total = usage['prompt_tokens'] + usage['response_tokens']
                print(f"   • {agent_name} ({usage['model']}):")
                print(f"     - Prompt: {usage['prompt_tokens']:,} tokens")
                print(f"     - Response: {usage['response_tokens']:,} tokens")
                print(f"     - Total: {agent_total:,} tokens")

        print(f"\n   Overall Total:")
        print(f"   • Prompt tokens: {total_prompt_tokens:,}")
        print(f"   • Response tokens: {total_response_tokens:,}")
        print(f"   • Total tokens: {total_tokens:,}")

        print("\n" + "="*50)
        print("✅ FINAL RESPONSE:")
        print("="*50)
        display(Markdown(final_response))
        print("="*50 + "\n")

    return final_response

# --- Initialize our Session Service ---
# This one service will manage all the different sessions in our notebook.
session_service = InMemorySessionService()
my_user_id = "adk_adventurer_001"

## Create the Agents

In [ ]:
# A tool to signal that the loop should terminate
COMPLETION_PHRASE = "The document meets all criteria."
def exit_loop(tool_context: ToolContext, final_document: str = ""):
  """Call this function ONLY when the document is approved, signaling the loop should end."""
  print(f"  [Tool Call] exit_loop triggered by {tool_context.agent_name}")

  # Print the final strategy document before exiting
  print("\n" + "="*60)
  print("🎯 FINAL APPROVED STRATEGY DOCUMENT:")
  print("="*60)

  if final_document:
    # TODO: Save the updated document back to Firestore
    print(final_document)
  else:
    print("⚠️ No final document provided to exit_loop function")

  print("="*60 + "\n")

  tool_context.actions.escalate = True
  return {"status": "Loop terminated successfully", "document_displayed": bool(final_document)}

# Search Agent: Uses Vertex AI Search to review internal business documents
DATASTORE_ID = "projects/ken-e-production/locations/global/collections/default_collection/dataStores/strategy-docs-acc-test-account_1755429117747"

internal_search_agent = Agent(
    name="internal_search_agent",
    model="gemini-2.5-flash",
    instruction="Answer questions using Vertex AI Search to find information from internal documents. Always cite sources when available.",
    description="Enterprise document search assistant with Vertex AI Search capabilities",
    tools=[VertexAiSearchTool(data_store_id=DATASTORE_ID)]
)

google_search_agent = Agent(
    name="google_search_agent",
    model="gemini-2.5-flash",
    instruction="Answer questions using Google Search to find information on the Internet. Always cite sources when available.",
    description="Internet research assistant with Google Search capabilities",
    tools=[google_search]
)

# Strategist Agent: Proposes an initial strategy doc
# This agent uses an advanced model, takes his time, and think carefully.
from google.adk.tools import AgentTool # Import AgentTool

strategist_agent = Agent(
    name="strategist_agent",
    model="gemini-1.5-pro-002",
    tools=[AgentTool(agent=internal_search_agent),AgentTool(agent=google_search_agent)], #[google_search], # Wrap search_agent in AgentTool
    description="A strategic thinking expert that creates or updates marketing strategy documents based on new information and best practice guidelines.",
    instruction="""
# ROLE & GOAL
You are a Strategic Marketing Expert who may be given one of the two following tasks:
1. Create a comprehensive business strategy document
2. Update an existing business strategy document based on new information

CRITICAL:
- You MUST output a complete JSON strategy document that follows the provided BEST PRACTICES schema exactly.

# TOOLS
- internal_search_agent: Searches through internal business strategy documents (including the business plan)
- google_search_agent: Uses Google search to find relevant information on the Internet

# YOUR TASK
You will receive several inputs in your conversation:
- A query describing what to do
- BEST PRACTICES: A JSON schema that defines the exact structure your output must follow
- NEW INFORMATION: Details about the company to research and analyze
- THE EXISTING STRATEGY DOCUMENT: You will only receive this if you are required to update an existing document. Otherwise you will know to create a new document.

# PERSONA
You are an expert strategist, meticulous in your analysis and precise in your writing. You ensure all outputs are professional, robust, and strictly adhere to provided guidelines. You are a critical thinker who can synthesize disparate information into a coherent strategic plan.

# PROCESS
You must follow this logic precisely:

1. **Analyze All Inputs:** Begin by thoroughly reading and understanding the query and all provided documents (`NEW INFORMATION`, `BEST PRACTICES`, and `strategy_document` if available).

2. **Determine Workflow:** Evaluate whether an `strategy_document` was provided. This determines your next steps.

3.A. **Workflow A: Update Existing Document**
   - If an `strategy_document` is provided, your task is to update it.
   - Cross-reference the `NEW INFORMATION` with the `strategy_document`.
   - Identify all new concepts from the `NEW INFORMATION` that must be integrated.
   - Identify all concepts in the `strategy_document` that are now obsolete or irrelevant due to the `NEW INFORMATION` and must be removed.
   - Meticulously rewrite and restructure the document to integrate the new concepts and remove the obsolete ones, ensuring the final document is coherent and flows logically.

3.B. **Workflow B: Create New Document**
   - If no `strategy_document` is provided, your task is to create one from scratch.
   - **MANDATORY**: Use the `google_search` tool extensively to research each item defined in the BEST PRACTICES.
   - Search for multiple queries related to each section you need to complete.
   - Synthesize your research findings into a complete, new strategy document that is well referenced with the URL's of the sources.
   - **MANDATORY**: You MUST add references any time you insert information that was found through one of your search agents so that the source document can be reviewed later.

4. **Research Requirements:**
   - For EVERY section of the document that requires external information, you MUST search for it
   - **MANDATORY**: You MUST add references any time you insert information that was found through one of your search agents so that the source document can be reviewed later.
   - Use specific, targeted search queries like:
     * "[Company name] mission vision values"
     * "[Company name] financial performance revenue"
     * "[Company name] competitors industry analysis"
     * "[Industry] market size trends 2025"
   - If you cannot find information needed for a section, insert the text: "requires further research"

6. **Final Review and Formatting:**
   - This is the most critical step. Before providing your response, validate your entire draft against the `BEST PRACTICES`.
   - Ensure every section, heading, and requirement from the guide is perfectly represented in your output document.

# OUTPUT REQUIREMENTS
- Your response must be ONLY the complete JSON strategy document
- Your final output MUST be the complete and final strategy document with ALL sections filled out based on your research.
- The structure, sections, and formatting of your response MUST EXACTLY MATCH the specifications in the `BEST PRACTICES`.
- DO NOT include any conversational text, preambles, or explanations in your output. Your response should only be the document itself.
- DO NOT leave any placeholder text or "requires further research" statements.

Remember: Your sole job is to output the complete JSON strategy document. Nothing else.
""",
    output_key="updated_strategy_doc"
)

# Reviewer Agent (in loop): Critiques the strategy doc based on the criteria provided in the Reviewer Guidelines document
# This agent may be able to use a smaller model
reviewer_agent = Agent(
    name="reviewer_agent",
    model="gemini-1.5-flash-latest", # Changed model to gemini-1.5-flash-latest
    tools=[],
    description="An expert content editor that reviews a strategy document against a set of guidelines to ensure completeness and quality.",
    instruction="""
# ROLE & GOAL
You are an experienced and meticulous Content Editor. Your primary goal is to review a draft strategy document against a provided set of guidelines and determine if all criteria are met.

# INPUTS
You will receive three key pieces of information:
- BEST PRACTICES: A JSON schema that defines the exact structure your output must follow.
- updated_strategy_doc: The draft strategy document that requires your review.
- REVIEWER GUIDELINES: A document containing the specific rules and criteria you must use for the review.

# PROCESS
1.  Thoroughly read and completely understand all criteria listed in the `REVIEWER GUIDELINES` and `BEST PRACTICES`.
2.  Systematically compare the `updated_strategy_doc` against each guideline.
3.  For each guideline, determine if the document satisfies the requirement. If information is unavailble for a section, the text should read: "requires further research"
4.  After checking all guidelines, make a final decision.

# OUTPUT REQUIREMENTS
Your response must strictly follow one of two formats:

- **IF** any of the guidelines are not met, you MUST provide a constructive critique with specific, actionable instructions for what needs to be fixed.
    - Example: 'This document is missing a required section. Include a short summary of the objective.'
    - Example: 'The "Key Metrics" section lacks quantifiable KPIs. Please add at least three measurable key performance indicators.'

- **ELSE**, if the document perfectly meets all criteria in the guidelines, you MUST respond with the exact phrase: 'The document meets all criteria.'
""",
    output_key="criticism"
)

# Editor Agent (in loop): Refines the strategy doc or exits
# This agent may need an advanced model to make careful edits to the strategy document
editor_agent = Agent(
    name="editor_agent",
    model="gemini-1.5-flash-latest", # Changed model to gemini-1.5-flash-latest
    tools=[AgentTool(agent=internal_search_agent),AgentTool(agent=google_search_agent),exit_loop],
    description="An expert marketing agent that refines and edits strategy documents based on specific criticism to ensure they meet best practice standards.",
    instruction="""
# ROLE & GOAL
You are a Senior Marketing Strategist at a top-tier marketing agency. Your primary goal is to meticulously edit a draft marketing strategy document based on feedback from a reviewer.

# TOOLS
- internal_search_agent: Searches through internal business strategy documents (including the business plan)
- google_search_agent: Uses Google search to find relevant information on the Internet
- exit_loop: A tool that signals the loop should terminate

# CRITICAL INSTRUCTION
You MUST check the criticism first. If the criticism is exactly "The document meets all criteria." then you MUST immediately call the exit_loop tool with the current document and provide NO other text output.

# PROCESS
1. **Review the Strategy Document**: Begin by thoroughly reading and understanding the provided strategy document (`updated_strategy_doc`).

2. **Check Criticism First**: Read the criticism input carefully.

3. **Exit Condition**:
   - IF the criticism is exactly the text "The document meets all criteria." (without quotes), then:
     * Call the exit_loop tool immediately and pass the current updated_strategy_doc as the final_document parameter
     * Provide NO text output at all
     * Do NOT edit the document

3. **Edit Condition**:
   - IF the criticism contains any other text, then:
     * Attempt to apply the requested changes to the document
     * Use the BEST PRACTICES to understand the description for each section
     * **MANDATORY**: Use the `google_search_agent` tool extensively to find new information on the Internet where necessary. Search for multiple queries related to each section you need to complete and synthesize your findings. If you cannot find information needed for a section, YOU MUST insert the text: "requires further research". Do NOT add information to the document unless you are certain that it is correct.
     * Use the 'internal_search_agent' tool to review the businesses internal strategy documents (including the business plan).
     * **MANDATORY**: You MUST add references any time you insert information that was found through one of your search agents so that the source document can be reviewed later.
     * Return the fully revised document

# INPUTS
You will receive:
- `updated_strategy_doc`: The current version of the document
- `BEST PRACTICES`: The JSON schema guide for structure
- `criticism`: The feedback to address

# OUTPUT REQUIREMENTS
- If calling exit_loop: Call exit_loop(final_document=updated_strategy_doc) and provide NO text output
- If editing: Return ONLY the complete revised document (no explanations)
""",
    output_key="updated_strategy_doc"
)

# The LoopAgent orchestrates the critique-refine cycle
refinement_loop = LoopAgent(
    name="refinement_loop",
    sub_agents=[reviewer_agent, editor_agent],
    description="Passes the ‘updated_strategy_doc’ generated by the strategist_agent to the reviewer agent and enables up to 3 rounds of edits.",
    max_iterations=3
)

# The SequentialAgent puts it all together
iterative_strategy_agent = SequentialAgent(
    name="iterative_strategy_agent",
    sub_agents=[strategist_agent, refinement_loop],
    description="A workflow that iteratively creates and refines a strategy document."
)

# The master dictionary of all our executable agents and workflows
worker_agents = {
    "iterative_strategy_agent": iterative_strategy_agent
}

## Test the Agents

## Mock Best Practice documents, and Firestore data
- strategy_doc: The current version of the strategy document that will be edited if it has already been created and added to Firestore. If none exists a new document will be created.
- best_practices: The best practices for creating this type of strategy document.
- reviewer_guidelines: The guidelines that the reviewer agent must follow when critiquing this type of strategy document.
- new_information: The new information that should be considered when creating or reviewing the strategy document.

In [ ]:
query = ""
new_information = ""
strategy_doc = ""
best_practices = ""
reviewer_guidelines = ""

### Mock Business Strategy documents

In [ ]:
# Set mock Firestore data for Business Strategy docs
if(test_document == "business strategy"):

  # Set the query and new_information for creating a new Business Strategy doc
  if(test_action == "create"):
    query = "Create a comprehensive strategy document for the company specified in the provided information. Use your tool 'google_search_agent' to find relevant information about the business on the Internet. This document will serve as the foundation for downstream tactical marketing plans."
    new_information = """
      COMPANY TO ANALYZE: Intellipure
      Website: https://www.intellipure.com/
      Industry: Retail Trade [B2C] - Selling goods directly to consumers through stores, online, or other direct channels
      Customer Regions: United States
      Target Company for Strategy Document: Intellipure (an air purification technology company)

      RESEARCH INSTRUCTIONS: Use the company name 'Intellipure' in all your Google searches to gather comprehensive information about this specific company for the strategy document.
      """
    strategy_doc = None

  # Set the query and new_information for updating an existing Business Strategy doc
  else:
    query = "Update the existing strategy document with the new information provided. Focus on integrating the new information into the appropriate sections while maintaining the existing document structure."
    new_information = """Intellipure was founded in January 2017"""

    # Mock an existing strategy doc that should be modified in Firestore
    strategy_doc = """
      {
        "executiveSummary": "Intellipure's marketing strategy for 2024-2025 prioritizes expanding market share and driving revenue growth within the U.S. air purification market. This strategy leverages digital channels, enhances customer engagement, and strengthens brand positioning.  Key strategic priorities include enhancing the customer experience, developing targeted marketing campaigns focusing on health-conscious consumers and allergy sufferers, and expanding into new market segments such as commercial spaces. The plan addresses challenges of increasing competition by emphasizing data-driven decision-making and agile adaptation to market trends. Financial projections indicate a targeted revenue increase of 15% by the end of 2025, based on increased market penetration and successful targeted marketing campaigns.",
        "companyOverview": {
          "historyAndBackground": "Intellipure's founding date and early history require further research.  However, the company has rapidly established itself as a provider of advanced air purification solutions.  Key milestones in its development will be included here once research is complete.  Sources will be cited.  [To be completed with specific details and citations]",
          "missionVisionValues": "Intellipure's mission is to provide innovative and high-quality air purification solutions that improve indoor air quality and promote healthier living environments. The company envisions becoming a leading provider of advanced air purification technologies, trusted for its commitment to customer satisfaction and environmental responsibility.  Its core values include innovation, customer focus, quality, and sustainability.",
          "leadershipAndOrganization": "Information on Intellipure's leadership team, organizational structure, and company size requires further research.  Once obtained, details including names of key personnel, organizational chart, and employee count will be provided here with sources cited. [To be completed with specific details and citations]",
          "brandAndCustomerBase": "Intellipure's brand is associated with advanced technology, high-quality performance, and a commitment to enhancing indoor air quality.  Its primary customer base consists of health-conscious consumers, allergy sufferers, and individuals seeking premium air purification solutions for residential use.  Research is needed to determine the precise size of Intellipure's customer base, its demographics, and the brand's overall market perception. Sources will be cited. [To be completed with specific details and citations]"
        },
        "productsAndServices": {
          "productServicePortfolio": "Intellipure's product portfolio includes a range of high-performance air purifiers designed for various residential settings.  Specific product models, features, and key selling points need to be researched and added here.  Details on the target customer segments for each product line will also be included.  Sources will be cited. [To be completed with specific details and citations]",
          "valueProposition": "Intellipure's value proposition centers on providing superior indoor air quality through innovative and effective air purification solutions. The company's products offer several key benefits, including improved respiratory health, reduced allergy symptoms, and enhanced overall well-being.  The premium quality of the products and advanced features justify the higher price point.",
          "competitivePositioning": "Intellipure positions itself as a premium provider of advanced air purification technology. The company differentiates itself from competitors by emphasizing its innovative filtration technologies, superior performance, and commitment to customer satisfaction. This premium positioning allows for a higher price point and targets a customer base seeking advanced features and superior quality.",
          "pricingModel": "Intellipure's pricing model requires further research.  Once the strategy is confirmed, this section will describe whether it's premium, competitive, or value-based, along with the rationale behind the pricing structure and how it aligns with the target market and competitive landscape. Sources will be cited. [To be completed with specific details and citations]"
        },
        "marketAndIndustryAnalysis": {
          "industryOverviewAndTrends": "The air purification industry is experiencing significant growth driven by increasing awareness of indoor air quality issues, rising concerns about respiratory health, and growing demand for advanced purification technologies.  Data on the size and growth rate of the industry will be added here, along with credible sources (market research reports, industry publications). [To be completed with specific details and citations]",
          "totalAddressableMarketTAM": "The total addressable market for residential air purifiers in the United States is substantial and growing, with opportunities for expansion into commercial and industrial sectors.  Once research is completed, this section will provide a quantified estimate of the TAM for Intellipure's products, along with the methodology used and sources cited. [To be completed with specific details and citations]",
          "competitiveLandscape": "Intellipure's main competitors include established players such as [Competitor A], [Competitor B], and [Competitor C], as well as emerging companies offering innovative air purification solutions.  A detailed competitive analysis outlining each competitor's strengths, weaknesses, market share, and competitive strategies will be provided once research is complete and sources will be cited. [To be completed with specific details and citations]",
          "industryForcesAnalysis": "A complete Porter's Five Forces analysis will be conducted for Intellipure's industry once research is complete to assess the competitive intensity, bargaining power of suppliers and buyers, threat of substitutes, and barriers to entry. [To be completed with specific details and citations]"
        },
        "externalEnvironmentAnalysisPESTEL": {
          "political": "Government regulations regarding indoor air quality and emissions standards will be researched and detailed here.  Sources will be cited. [To be completed with specific details and citations]",
          "economic": "Macroeconomic factors such as consumer spending and disposable income will be analyzed and included here.  Sources will be cited. [To be completed with specific details and citations]",
          "social": "Consumer preferences for health and wellness, environmental awareness, and demand for smart home technology will be assessed and included here.  Sources will be cited. [To be completed with specific details and citations]",
          "technological": "Technological advancements in air filtration, sensor technology, and IoT integration will be reviewed and included here.  Sources will be cited. [To be completed with specific details and citations]",
          "environmental": "Sustainability concerns and regulations related to manufacturing and product disposal will be evaluated and included here.  Sources will be cited. [To be completed with specific details and citations]",
          "legal": "Relevant laws and regulations regarding product safety, labeling, and emissions will be analyzed and included here.  Sources will be cited. [To be completed with specific details and citations]"
        },
        "internalOperationsAndBusinessModel": {
          "businessModel": "Intellipure's business model centers on the design, manufacturing, and sale of high-performance air purifiers directly to consumers.  Revenue is generated through direct sales and possibly online channels.  The cost structure comprises manufacturing, marketing, distribution, and administrative expenses.  Once researched, more detail will be provided here. [To be completed with specific details and citations]",
          "valueChainAndSupplyChain": "The company's value chain encompasses product design, sourcing of components, manufacturing, marketing and sales, and customer service.  Supply chain management practices will be examined to ensure efficient procurement and production.  More details will be added here after research is complete. [To be completed with specific details and citations]",
          "operationalEfficiency": "Operational efficiency metrics will be reviewed, focusing on production costs, inventory management, and supply chain optimization.  Further research and details will be provided here. [To be completed with specific details and citations]",
          "salesChannelsAndDistribution": "Sales channels include direct sales, e-commerce, and potentially partnerships with retailers.  The efficiency of distribution channels will be evaluated. More details will be provided here after research is complete. [To be completed with specific details and citations]"
        },
        "marketingAndCustomerStrategy": {
          "targetMarketAndCustomerSegmentation": "Intellipure's target market includes health-conscious consumers, allergy sufferers, families with young children, and individuals living in areas with poor air quality.  Once complete, market segmentation will be refined to identify specific customer personas and their needs. [To be completed with specific details and citations]",
          "brandPositioning": "The brand will be positioned as a premium provider of advanced air purification solutions, emphasizing superior technology, performance, and customer satisfaction.",
          "customerAcquisitionStrategies": "Customer acquisition strategies will include digital marketing (SEO, SEM, social media), content marketing, influencer collaborations, and potentially partnerships with healthcare professionals or allergy organizations.  Specific tactics and budget allocations will be detailed once research is complete. [To be completed with specific details and citations]",
          "digitalPresenceAndSocialMedia": "A strong online presence will be maintained, including a user-friendly website, engaging social media content, and targeted online advertising campaigns.",
          "customerExperienceAndRetention": "A positive customer experience will be fostered through excellent customer service, product warranties, and loyalty programs."
        },
        "financialPerformanceAndAnalysis": {
          "revenueAndGrowth": "Intellipure's revenue growth will be analyzed over recent years, comparing it to industry averages.  Financial data will be included here once research is complete and sources will be cited. [To be completed with specific details and citations]",
          "profitability": "Profitability margins will be reviewed, including gross, operating, and net profit margins.  Financial data will be included here once research is complete and sources will be cited. [To be completed with specific details and citations]",
          "cashFlowAndFinancialStability": "Cash flow and financial stability will be assessed, including debt levels and liquidity ratios.  Financial data will be included here once research is complete and sources will be cited. [To be completed with specific details and citations]",
          "keyFinancialRatios": "Key financial ratios such as ROI, LTV/CAC, and others relevant to the business will be calculated and compared to industry benchmarks.  Financial data will be included here once research is complete and sources will be cited. [To be completed with specific details and citations]",
          "financialOutlook": "The financial outlook will be projected based on expected revenue growth and market conditions.  Financial data will be included here once research is complete and sources will be cited. [To be completed with specific details and citations]"
        },
        "swotAnalysis": {
          "strengths": [
            "Advanced air purification technology",
            "High-quality products",
            "Strong brand reputation",
            "Commitment to customer satisfaction"
          ],
          "weaknesses": [
            "Limited brand awareness in some segments",
            "Relatively small market share",
            "Potential for increased competition"
          ],
          "opportunities": [
            "Expanding into new market segments (commercial/industrial)",
            "Leveraging technological advancements",
            "Strategic partnerships with retailers or healthcare providers"
          ],
          "threats": [
            "Increasing competition from established and emerging players",
            "Economic downturns impacting consumer spending",
            "Changing consumer preferences and trends"
          ]
        },
        "strategicRecommendationsAndFutureOutlook": {
          "keyIssuesIdentified": "Intellipure faces the challenge of expanding market share in a competitive industry while building stronger brand awareness.  Capitalizing on technological advancements and expanding into new market segments is crucial for long-term growth.",
          "recommendations": [
            {
              "recommendation": "Invest in targeted digital marketing campaigns to increase brand awareness and drive sales.",
              "justification": "Digital marketing offers cost-effective and efficient ways to reach specific target segments.",
              "implementationNotes": "Allocate sufficient budget to SEO, SEM, and social media marketing.  Develop compelling content and track campaign performance.  Timelines and KPIs will be added after further research. [To be completed with specific details and citations]"
            },
            {
              "recommendation": "Expand product portfolio to include solutions for commercial and industrial settings.",
              "justification": "Untapped market potential exists in commercial and industrial air purification.",
              "implementationNotes": "Conduct market research to identify target segments and develop tailored product offerings. Timelines and KPIs will be added after further research. [To be completed with specific details and citations]"
            },
            {
              "recommendation": "Develop strategic partnerships to expand reach and market penetration.",
              "justification": "Partnerships offer access to new customer bases and marketing channels.",
              "implementationNotes": "Identify potential partners in retail, construction, or healthcare industries. Timelines and KPIs will be added after further research. [To be completed with specific details and citations]"
            }
          ],
          "futureOutlook": "With the successful execution of this strategy, Intellipure is projected to experience significant growth in revenue and market share. The company will become a leading provider of advanced air purification solutions, known for its superior products, strong brand reputation, and commitment to customer satisfaction."
        },
        "references": []
      }
      """

  # Set mock best practices guide from Firestore
  # These are passed regardless of creating a new doc or modifying an existing doc
  best_practices = """{
    "$schema": "http://json-schema.org/draft-07/schema#",
    "title": "Comprehensive Business Strategy Report",
    "description": "A standardized JSON structure for generating a comprehensive business strategy report. All generated documents must conform to this schema.",
    "type": "object",
    "required": [
      "executiveSummary",
      "companyOverview",
      "productsAndServices",
      "marketAndIndustryAnalysis",
      "externalEnvironmentAnalysisPESTEL",
      "internalOperationsAndBusinessModel",
      "marketingAndCustomerStrategy",
      "financialPerformanceAndAnalysis",
      "swotAnalysis",
      "strategicRecommendationsAndFutureOutlook"
    ],
    "properties": {
      "executiveSummary": {
        "type": "string",
        [cite_start]"description": "A high-level summary of the company's situation, strategic direction, and key findings or recommendations. This should be written last but placed first. [cite: 3, 4, 5]"
      },
      "companyOverview": {
        "type": "object",
        [cite_start]"description": "Introduces the company's identity and background. [cite: 6, 7]",
        "required": [
          "historyAndBackground",
          "missionVisionValues",
          "leadershipAndOrganization",
          "brandAndCustomerBase"
        ],
        "properties": {
          "historyAndBackground": {
            "type": "string",
            [cite_start]"description": "Founding details, major milestones, and evolution of the business. [cite: 8]"
          },
          "missionVisionValues": {
            "type": "string",
            [cite_start]"description": "The company's core purpose, long-term vision, and guiding principles. [cite: 9]"
          },
          "leadershipAndOrganization": {
            "type": "string",
            [cite_start]"description": "Overview of the leadership team, organizational structure, and team size. [cite: 10]"
          },
          "brandAndCustomerBase": {
            "type": "string",
            [cite_start]"description": "The brand identity, reputation, and general characteristics of target customers. [cite: 11]"
          }
        }
      },
      "productsAndServices": {
        "type": "object",
        [cite_start]"description": "Details on the company's main products or services and how they drive value. [cite: 13, 14]",
        "required": [
          "productServicePortfolio",
          "valueProposition",
          "competitivePositioning",
          "pricingModel"
        ],
        "properties": {
          "productServicePortfolio": {
            "type": "string",
            [cite_start]"description": "A description of the flagship products, services, or product lines, including their features and uses. [cite: 15]"
          },
          "valueProposition": {
            "type": "string",
            [cite_start]"description": "The customer problems each product/service solves and the unique benefits provided. [cite: 16, 17]"
          },
          "competitivePositioning": {
            "type": "string",
            [cite_start]"description": "How each product is positioned relative to competitors' offerings, including unique selling points. [cite: 18]"
          },
          "pricingModel": {
            "type": "string",
            [cite_start]"description": "Overview of the pricing strategy (e.g., premium, subscription) and how it supports revenue goals. [cite: 20]"
          }
        }
      },
      "marketAndIndustryAnalysis": {
        "type": "object",
        [cite_start]"description": "An examination of the industry and market environment to understand external opportunities and threats. [cite: 22, 23]",
        "required": [
          "industryOverviewAndTrends",
          "totalAddressableMarketTAM",
          "competitiveLandscape",
          "industryForcesAnalysis"
        ],
        "properties": {
          "industryOverviewAndTrends": {
            "type": "string",
            [cite_start]"description": "The current state of the industry (size, growth rate) and important trends shaping the market. [cite: 24]"
          },
          "totalAddressableMarketTAM": {
            "type": "string",
            [cite_start]"description": "An estimate of the total potential customers or revenue available in the relevant sector. [cite: 25]"
          },
          "competitiveLandscape": {
            "type": "string",
            [cite_start]"description": "Identification and assessment of key industry players and direct competitors. [cite: 27, 28]"
          },
          "industryForcesAnalysis": {
            "type": "string",
            [cite_start]"description": "An analysis of industry structure using a framework like Porter's Five Forces. [cite: 29]"
          }
        }
      },
      "externalEnvironmentAnalysisPESTEL": {
        "type": "object",
        [cite_start]"description": "A PESTEL analysis examining macro-environmental factors that could impact the company's strategy. [cite: 32, 33]",
        "required": [
          "political",
          "economic",
          "social",
          "technological",
          "environmental",
          "legal"
        ],
        "properties": {
          "political": {
            "type": "string",
            [cite_start]"description": "Relevant government policies, political stability, and trade agreements. [cite: 34]"
          },
          "economic": {
            "type": "string",
            [cite_start]"description": "Macro-economic trends like inflation, economic growth, and employment levels. [cite: 35]"
          },
          "social": {
            "type": "string",
            [cite_start]"description": "Societal trends, demographic changes, and consumer behaviors that shape market needs. [cite: 36]"
          },
          "technological": {
            "type": "string",
            [cite_start]"description": "The impact of technological change, innovation, and R&D trends in the industry. [cite: 37]"
          },
          "environmental": {
            "type": "string",
            [cite_start]"description": "Environmental and sustainability factors, such as climate change and ecological regulations. [cite: 38]"
          },
          "legal": {
            "type": "string",
            [cite_start]"description": "Laws and regulations (e.g., industry regulations, labor laws) the company must comply with. [cite: 39]"
          }
        }
      },
      "internalOperationsAndBusinessModel": {
        "type": "object",
        [cite_start]"description": "An analysis of the internal workings of the company to identify strengths and weaknesses. [cite: 42, 43]",
        "required": [
          "businessModel",
          "valueChainAndSupplyChain",
          "operationalEfficiency",
          "salesChannelsAndDistribution"
        ],
        "properties": {
          "businessModel": {
            "type": "string",
            [cite_start]"description": "Explanation of how the company generates revenue and profit, including core revenue streams and cost structure. [cite: 44, 45]"
          },
          "valueChainAndSupplyChain": {
            "type": "string",
            [cite_start]"description": "Overview of the production or service delivery process, from sourcing inputs to distribution. [cite: 46, 47]"
          },
          "operationalEfficiency": {
            "type": "string",
            [cite_start]"description": "Assessment of the company's operational capabilities, efficiency metrics, and quality control. [cite: 49]"
          },
          "salesChannelsAndDistribution": {
            "type": "string",
            [cite_start]"description": "The channels through which the company sells and reaches customers (e.g., online, retail, B2B sales force). [cite: 52]"
          }
        }
      },
      "marketingAndCustomerStrategy": {
        "type": "object",
        [cite_start]"description": "Insights into how the company attracts, converts, and retains its customers. [cite: 57, 58]",
        "required": [
          "targetMarketAndCustomerSegmentation",
          "brandPositioning",
          "customerAcquisitionStrategies",
          "digitalPresenceAndSocialMedia",
          "customerExperienceAndRetention"
        ],
        "properties": {
          "targetMarketAndCustomerSegmentation": {
            "type": "string",
            [cite_start]"description": "Identification and description of the company's primary customer segments or personas. [cite: 59, 60]"
          },
          "brandPositioning": {
            "type": "string",
            [cite_start]"description": "How the company positions its brand in the market relative to competitors. [cite: 62]"
          },
          "customerAcquisitionStrategies": {
            "type": "string",
            [cite_start]"description": "The marketing and sales tactics used to reach new customers, including the marketing mix. [cite: 64, 65]"
          },
          "digitalPresenceAndSocialMedia": {
            "type": "string",
            [cite_start]"description": "An overview of the company's online presence, including website, SEO/SEM, and social media. [cite: 66, 67, 69]"
          },
          "customerExperienceAndRetention": {
            "type": "string",
            [cite_start]"description": "How the company manages customer relationships to encourage repeat business and loyalty. [cite: 71]"
          }
        }
      },
      "financialPerformanceAndAnalysis": {
        "type": "object",
        [cite_start]"description": "A review of the company's financial health and trajectory. [cite: 75, 76]",
        "required": [
          "revenueAndGrowth",
          "profitability",
          "cashFlowAndFinancialStability",
          "keyFinancialRatios",
          "financialOutlook"
        ],
        "properties": {
          "revenueAndGrowth": {
            "type": "string",
            [cite_start]"description": "The company's revenues and growth rate over recent years, compared to industry averages. [cite: 77, 79]"
          },
          "profitability": {
            "type": "string",
            [cite_start]"description": "Examination of margins and profits (gross, operating, net) and how they have trended. [cite: 80]"
          },
          "cashFlowAndFinancialStability": {
            "type": "string",
            [cite_start]"description": "Summary of the company's cash flow situation, debt levels, and overall financial risk. [cite: 83, 85]"
          },
          "keyFinancialRatios": {
            "type": "string",
            [cite_start]"description": "A few critical financial ratios or KPIs (e.g., ROI, LTV/CAC) compared to industry benchmarks. [cite: 89, 91]"
          },
          "financialOutlook": {
            "type": "string",
            [cite_start]"description": "Any management guidance or analyst projections for future financial performance. [cite: 92]"
          }
        }
      },
      "swotAnalysis": {
        "type": "object",
        [cite_start]"description": "A synthesis of the internal and external analyses into a quadrant framework. [cite: 96, 97]",
        "required": [
          "strengths",
          "weaknesses",
          "opportunities",
          "threats"
        ],
        "properties": {
          "strengths": {
            "type": "array",
            [cite_start]"description": "Internal capabilities, resources, or advantages that give the company an edge. [cite: 98]",
            "items": {
              "type": "string"
            }
          },
          "weaknesses": {
            "type": "array",
            [cite_start]"description": "Internal limitations or gaps that hinder performance. [cite: 100]",
            "items": {
              "type": "string"
            }
          },
          "opportunities": {
            "type": "array",
            [cite_start]"description": "External chances to improve or grow the business, based on industry trends or market gaps. [cite: 102]",
            "items": {
              "type": "string"
            }
          },
          "threats": {
            "type": "array",
            [cite_start]"description": "External factors that could negatively impact the company's performance. [cite: 104]",
            "items": {
              "type": "string"
            }
          }
        }
      },
      "strategicRecommendationsAndFutureOutlook": {
        "type": "object",
        [cite_start]"description": "Conclusions from the analysis and suggested strategic actions. [cite: 106, 107]",
        "required": [
          "keyIssuesIdentified",
          "recommendations",
          "futureOutlook"
        ],
        "properties": {
          "keyIssuesIdentified": {
            "type": "string",
            [cite_start]"description": "A brief recap of the most critical challenges or strategic issues the company is facing. [cite: 109]"
          },
          "recommendations": {
            "type": "array",
            [cite_start]"description": "Specific, actionable strategies to address key issues and capitalize on opportunities. [cite: 113]",
            "items": {
              "type": "object",
              "required": ["recommendation", "justification", "implementationNotes"],
              "properties": {
                "recommendation": {
                  "type": "string",
                  "description": "The specific tactical move being recommended."
                },
                "justification": {
                  "type": "string",
                  "description": "How this recommendation ties back to findings in the analysis (e.g., leveraging a strength to seize an opportunity)."
                },
                "implementationNotes": {
                  "type": "string",
                  [cite_start]"description": "Brief notes on required changes in structure, resources, or policies to support the new strategy. [cite: 115, 116]"
                }
              }
            }
          },
          "futureOutlook": {
            "type": "string",
            [cite_start]"description": "A closing perspective on the company's projected future, assuming the strategy is successfully executed. [cite: 117, 118]"
          }
        }
      },
      "references": {
          "type": "array",
          [cite_start]"description": "A list of sources for facts, figures, and research used in the report to enhance credibility. [cite: 130, 131]",
          "items": {
              "type": "string"
          }
      }
    }
  }
  """

  # Set mock reviewer_guidelines from Firestore
  # These are the same regardless of creating a new doc or modifyin an existing one
  reviewer_guidelines = """
    ### \#\# Critique Instructions

    **1. Structural Validation:**

      * Validate the `updated_strategy_doc` against the BEST PRACTICES.
      * Identify any **missing** required keys.
      * Identify any keys with the **incorrect data type** (e.g., a `string` where an `array` is required).
      * Identify any **extra keys** present in the document that are not defined in the schema.

    **2. Content Completeness Check:**

      * For every field, verify that the provided content is **present and non-trivial**.
      * Flag any fields that are empty strings (`""`), `null`, or contain only placeholder text (e.g., "TBD", "[to be completed]", "Insert text here").
      * Flag any descriptions or analyses that are nonsensically short (e.g., a single word where a paragraph is expected), as this indicates incomplete generation.

    **3. Generate Actionable Feedback:**

      * For each error you uncover, provide a clear description of the issue and a specific suggestion for how to fix it.
      * Compile all findings into a single JSON object, following the **Output Format** specified below precisely. If there are no errors, the `errors` array should be empty and `isValid` should be `true`.

    -----

    ### \#\# Output Format (Strict)

    Your entire response must be a single JSON object matching this structure:

    ```json
    {
      "critiqueReport": {
        "isValid": false,
        "errors": [
          {
            "errorType": "STRUCTURAL_ERROR | CONTENT_ERROR",
            "fieldPath": "path.to.the.problematic.field",
            "issue": "A clear, concise description of what is wrong.",
            "suggestion": "A specific, actionable instruction on how to correct the error."
          }
        ]
      }
    }
    ```

    -----

    ### \#\# Example Critique

    **Example Issue:** The `valueProposition` key is missing from the `productsAndServices` object.

    **Your Corresponding Output Error Object:**

    ```json
    {
      "errorType": "STRUCTURAL_ERROR",
      "fieldPath": "productsAndServices.valueProposition",
      "issue": "The required key 'valueProposition' is missing from the object.",
      "suggestion": "Add the required key 'valueProposition' and provide a string value explaining the unique benefits the product solves for customers."
    }
    ```

    **Example Issue:** The `historyAndBackground` field contains only the word "History".

    **Your Corresponding Output Error Object:**

    ```json
    {
      "errorType": "CONTENT_ERROR",
      "fieldPath": "companyOverview.historyAndBackground",
      "issue": "The content for this field is incomplete and lacks detail.",
      "suggestion": "Expand this field to include details about the company's founding, major milestones, and its evolution over time."
    }
    ```
    """

### Mock Competitive Strategy documents

In [ ]:
# Set mock Firestore data for Business Strategy docs
if(test_document == "competitive strategy"):

  # Set the query and new_information for creating a new Business Strategy doc
  if(test_action == "create"):
    query = "Create a comprehensive competitive strategy document for the company specified in the provided information. Follow the provided best practices carefully. Use your tool 'internal_search_agent' to learn about the business strategy, and use your tool 'google_search_agent' to conduct research on the business and its competitors online. This document will serve as the foundation for downstream tactical marketing plans."
    new_information = """
      COMPANY TO ANALYZE: Intellipure
      Website: https://www.intellipure.com/
      Industry: Retail Trade [B2C] - Selling goods directly to consumers through stores, online, or other direct channels
      Customer Regions: United States
      Target Company for Strategy Document: Intellipure (an air purification technology company)

      RESEARCH INSTRUCTIONS: Use the company name 'Intellipure' in all your Google searches to gather comprehensive information about this specific company for the strategy document.
      """
    strategy_doc = None

  # Set the query and new_information for updating an existing Business Strategy doc
  else:
    query = "Update the existing strategy document with the new information provided. Focus on integrating the new information into the appropriate sections while maintaining the existing document structure."
    new_information = """A new company called 'Filter Buddy' has entered the market to compete with Intellipure's B2C product."""

    # Mock an existing strategy doc that should be modified in Firestore
    strategy_doc = """
      {
      "strategicRecommendations": {
        "monitoringPlan": "Continuously monitor competitor websites, industry publications, and social media for new product launches, pricing changes, and marketing campaigns. Track market share trends and customer reviews to identify emerging threats and opportunities. Conduct quarterly competitor analysis reports.",
        "leveragingStrengths": "Focus on highlighting Intellipure's patented DFS technology and its effectiveness in removing ultrafine particles. Emphasize the medical-grade filtration and health benefits, targeting health-conscious consumers and those with allergies or respiratory sensitivities. Leverage partnerships with healthcare professionals and institutions to build credibility and trust.",
        "productAndServiceStrategy": "Explore expanding the product line to include portable air purifiers for travel or smaller spaces. Develop smart home integration features and personalized air quality monitoring through a mobile app. Offer subscription-based filter replacement services for added convenience.",
        "marketPositioning": "Position Intellipure as the premium air purification solution for health and wellness, emphasizing superior performance and medical-grade filtration. Target specific customer segments, such as allergy sufferers, families with young children, and individuals concerned about indoor air quality. Highlight the long-term health benefits and cost savings compared to cheaper alternatives.",
        "addressingCompetitiveGaps": "Invest in marketing and brand awareness campaigns to improve visibility and reach a wider audience. Expand distribution channels to include online marketplaces and retail partnerships. Offer competitive pricing and financing options to address price sensitivity.",
        "marketingAndSalesStrategy": "Develop targeted advertising campaigns highlighting the unique benefits of Intellipure's technology. Partner with health influencers and bloggers to reach relevant audiences. Offer educational content about indoor air quality and the importance of effective filtration. Implement a customer loyalty program to incentivize repeat purchases and referrals.",
        "prioritizedOpportunities": "Focus on expanding market share in the residential and healthcare sectors. Target new customer segments, such as businesses and educational institutions. Develop strategic partnerships with HVAC companies and indoor air quality specialists."
      },
      "competitiveLandscape": {
        "marketShareAndPositioning": [
          {
            "marketSharePercentage": "requires further research",
            "competitorName": "Austin Air",
            "positioning": "Focuses on durability and long-lasting filters."
          },
          {
            "marketSharePercentage": "requires further research",
            "competitorName": "IQAir",
            "positioning": "Known for high-performance air purifiers with advanced filtration."
          },
          {
            "marketSharePercentage": "requires further research",
            "competitorName": "Alen",
            "positioning": "Offers a range of purifiers with different features and price points."
          },
          {
            "marketSharePercentage": "requires further research",
            "competitorName": "Molekule",
            "positioning": "Emphasizes innovative air purification technology."
          }
        ],
        "competitorSegmentation": {
          "strategicGroups": "The market can be segmented into groups based on pricing (low-cost vs. premium), technology (HEPA vs. PECO vs. DFS), and target market (residential vs. commercial)."
        },
        "competitorIdentification": [
          {
            "type": "Direct",
            "name": "Austin Air"
          },
          {
            "type": "Direct",
            "name": "IQAir"
          },
          {
            "type": "Direct",
            "name": "Alen"
          },
          {
            "type": "Direct",
            "name": "Molekule"
          },
          {
            "type": "Indirect",
            "name": "Honeywell"
          },
          {
            "type": "Indirect",
            "name": "Coway"
          }
        ]
      },
      "comparativeAnalysis": {
        "comparisonMatrix": "requires further research",
        "brandPositioningMap": {
          "description": "A brand positioning map visualizing competitors based on price and performance would show Intellipure positioned in the premium segment with high performance.",
          "axisY": "Performance",
          "axisX": "Price",
          "competitorPositions": [
            {
              "y": "High",
              "name": "IQAir",
              "x": "High"
            },
            {
              "y": "Medium",
              "name": "Austin Air",
              "x": "Medium"
            },
            {
              "y": "Medium",
              "name": "Alen",
              "x": "Medium"
            },
            {
              "y": "High",
              "name": "Molekule",
              "x": "High"
            },
            {
              "y": "High",
              "name": "Intellipure",
              "x": "High"
            }
          ]
        },
        "featureAndCapabilityGaps": "requires further research"
      },
      "marketAndIndustryOverview": {
        "pestelAnalysis": {
          "technological": "Advancements in filtration technology and sensor development are driving innovation in the air purification market. Smart home integration and IoT connectivity are becoming increasingly important.",
          "political": "Government regulations and air quality standards influence the demand for air purifiers.",
          "social": "Growing awareness of indoor air pollution and its health effects is driving consumer demand for air purifiers. Concerns about allergens, viruses, and other airborne contaminants are also contributing factors.",
          "environmental": "Climate change and increasing air pollution levels are driving the need for effective air purification solutions.",
          "economic": "Economic growth and rising disposable incomes in developing countries are creating new market opportunities for air purifier manufacturers.",
          "legal": "Regulations related to product safety and energy efficiency impact the design and manufacturing of air purifiers."
        },
        "marketDefinition": {
          "growthRate": "The global air purifier market is expected to grow at a CAGR of over 10% during the forecast period.",
          "size": "requires further research",
          "keyTrends": "Increasing adoption of smart home technology, growing demand for portable and personal air purifiers, and rising awareness of the health benefits of clean indoor air are some of the key trends shaping the market."
        },
        "portersFiveForces": {
          "threatOfNewEntrants": "The threat of new entrants is moderate due to the relatively low barriers to entry. However, establishing brand reputation and gaining market share can be challenging.",
          "supplierPower": "The bargaining power of suppliers is moderate, as there are a number of suppliers providing components for air purifiers.",
          "competitiveRivalry": "The competitive rivalry among existing players is high, with companies competing on price, features, and technology.",
          "threatOfSubstitutes": "The threat of substitutes is low, as there are limited alternatives to air purifiers for effectively removing airborne pollutants.",
          "buyerPower": "The bargaining power of buyers is moderate, with consumers having a variety of options to choose from."
        },
        "keySuccessFactors": "Key success factors in the air purifier market include technological innovation, product performance, brand reputation, distribution network, and customer service."
      },
      "analysisObjectivesAndScope": {
        "objectives": "The objectives of this analysis are to understand the competitive landscape, identify key competitors, assess their strengths and weaknesses, and develop strategic recommendations for Intellipure to gain a competitive advantage.",
        "scope": {
          "competitorTypes": "This analysis includes direct and indirect competitors in the residential and commercial air purifier market.",
          "geographicMarket": "The geographic scope of this analysis is the United States."
        }
      },
      "internalSWOTAnalysis": {
        "opportunities": "Growing demand for high-performance air purifiers, increasing awareness of indoor air quality, and the rising adoption of smart home technology present significant opportunities for Intellipure.",
        "weaknesses": "Limited brand awareness and distribution network compared to established competitors are key weaknesses for Intellipure.",
        "strengths": "Intellipure's patented DFS technology, medical-grade filtration, and focus on health and wellness are key strengths that differentiate the company from competitors.",
        "threats": "Intense competition from established players, price sensitivity among consumers, and the potential for new disruptive technologies pose threats to Intellipure."
      },
      "detailedCompetitorProfiles": [
        {
          "background": {
            "ownership": "requires further research",
            "scale": "requires further research",
            "history": "requires further research"
          },
          "competitorName": "Austin Air",
          "targetMarket": "Residential and commercial customers looking for durable and long-lasting air purifiers.",
          "pricingStrategy": "Premium pricing based on the durability and long-life filters.",
          "weaknesses": [
            "Limited product variety",
            "Traditional design"
          ],
          "marketingAndPromotion": "Focuses on product features and benefits, emphasizing long-lasting filters and durability. Utilizes online and offline marketing channels.",
          "distributionChannels": "Online retailers, distributors, and dealers.",
          "productsAndValueProposition": "Offers a range of air purifiers with long-lasting filters, suitable for various room sizes and applications.",
          "strengths": [
            "Durable and long-lasting filters",
            "Strong reputation for quality"
          ]
        },
        {
          "background": {
            "ownership": "requires further research",
            "scale": "requires further research",
            "history": "requires further research"
          },
          "competitorName": "IQAir",
          "targetMarket": "Residential and commercial customers seeking high-performance air purification solutions.",
          "pricingStrategy": "Premium pricing reflecting the high performance and advanced filtration technology.",
          "weaknesses": [
            "High price point",
            "Complex product offerings"
          ],
          "marketingAndPromotion": "Emphasizes high performance and advanced technology, targeting health-conscious consumers. Uses online and offline marketing channels.",
          "distributionChannels": "Online retailers, distributors, and dealers.",
          "productsAndValueProposition": "Offers a range of high-performance air purifiers with advanced filtration technology, suitable for various applications.",
          "strengths": [
            "High-performance filtration",
            "Advanced technology"
          ]
        },
        {
          "background": {
            "ownership": "requires further research",
            "scale": "requires further research",
            "history": "requires further research"
          },
          "competitorName": "Alen",
          "targetMarket": "Residential customers looking for a variety of air purifiers with different features and price points.",
          "pricingStrategy": "Offers a range of price points to cater to different customer segments.",
          "weaknesses": [
            "Varying product quality",
            "Less focus on advanced technology"
          ],
          "marketingAndPromotion": "Focuses on product features and benefits, highlighting different options for various needs and budgets. Uses online and offline marketing channels.",
          "distributionChannels": "Online retailers, distributors, and dealers.",
          "productsAndValueProposition": "Offers a range of air purifiers with different features and price points, catering to a broader consumer base.",
          "strengths": [
            "Wide product selection",
            "Variety of price points"
          ]
        },
        {
          "background": {
            "ownership": "requires further research",
            "scale": "requires further research",
            "history": "requires further research"
          },
          "competitorName": "Molekule",
          "targetMarket": "Residential customers seeking innovative air purification technology.",
          "pricingStrategy": "Premium pricing based on the innovative technology and unique features.",
          "weaknesses": [
            "High price point",
            "Limited product variety"
          ],
          "marketingAndPromotion": "Emphasizes the innovative technology and design, targeting tech-savvy consumers. Primarily utilizes online marketing channels.",
          "distributionChannels": "Online retailers and direct sales.",
          "productsAndValueProposition": "Offers innovative air purifiers with unique technology, claiming to destroy pollutants rather than just trapping them.",
          "strengths": [
            "Innovative technology",
            "Unique value proposition"
          ]
        }
      ],
      "executiveSummary": {
        "strategicRecommendations": "Intellipure should leverage its patented DFS technology and medical-grade filtration to position itself as the premium air purification solution for health and wellness. Focus on targeted marketing campaigns, strategic partnerships, and product line expansion to capture a larger market share.",
        "keyFindings": "The air purifier market is experiencing significant growth driven by increasing awareness of indoor air quality and the health benefits of clean air. Intellipure faces competition from established players offering a range of products and technologies. By focusing on its core strengths and addressing key weaknesses, Intellipure can effectively compete in this growing market."
      }
      """

  # Set mock best practices guide from Firestore
  # These are passed regardless of creating a new doc or modifying an existing doc
  best_practices = """{
    "_collection": "strategy_doc_guides",
    "_document_id": "competitive_strategy_best_practices",
    "strategicRecommendations": {
      "monitoringPlan": "Propose a plan for continuous monitoring of competitors and market dynamics.",
      "leveragingStrengths": "Outline how to capitalize on unique strengths to build a competitive advantage.",
      "productAndServiceStrategy": "Suggest product development or enhancements based on identified market needs and competitor offerings.",
      "marketPositioning": "Recommend how to refine brand messaging and positioning to stand out in the market.",
      "addressingCompetitiveGaps": "Propose specific actions to remedy weaknesses and close gaps with competitors.",
      "marketingAndSalesStrategy": "Recommend marketing and sales tactics to effectively communicate value and gain a competitive edge.",
      "prioritizedOpportunities": "Identify the most promising opportunities to pursue, aligned with company strengths."
    },
    "competitiveLandscape": {
      "marketShareAndPositioning": [
        {
          "marketSharePercentage": "Number",
          "competitorName": "Name",
          "positioning": "Describe how the competitor differentiates itself (e.g., price leader, quality leader, innovator)."
        }
      ],
      "competitorSegmentation": {
        "strategicGroups": "Categorize competitors into groups based on their strategic approach (e.g., low-cost providers vs. premium brands)."
      },
      "competitorIdentification": [
        {
          "type": "Direct/Indirect/Emerging",
          "name": "Competitor Name"
        }
      ]
    },
    "comparativeAnalysis": {
      "comparisonMatrix": "Provide a summary comparing competitors across key criteria like pricing, product range, quality, and customer ratings.",
      "brandPositioningMap": {
        "description": "Visualize competitor positions based on two key attributes (e.g., Price vs. Quality). Identify clusters and open 'white space'.",
        "axisY": "Attribute 2 (e.g., Quality)",
        "axisX": "Attribute 1 (e.g., Price)",
        "competitorPositions": [
          {
            "y": "Low/Medium/High",
            "name": "Competitor Name",
            "x": "Low/Medium/High"
          },
          {
            "y": "Low/Medium/High",
            "name": "Our Company",
            "x": "Low/Medium/High"
          }
        ]
      },
      "featureAndCapabilityGaps": "Identify unmet customer needs or market gaps not addressed by competitors."
    },
    "marketAndIndustryOverview": {
      "pestelAnalysis": {
        "technological": "Analyze technological advancements and disruptions.",
        "political": "Analyze political factors impacting the industry.",
        "social": "Analyze social and demographic trends impacting the industry.",
        "environmental": "Analyze environmental factors and regulations.",
        "economic": "Analyze economic conditions impacting the industry.",
        "legal": "Analyze legal and regulatory changes."
      },
      "marketDefinition": {
        "growthRate": "Provide the projected market growth rate.",
        "size": "State the current market size and value.",
        "keyTrends": "Describe significant market trends and shifting consumer behaviors."
      },
      "portersFiveForces": {
        "threatOfNewEntrants": "Assess the threat of new companies entering the market.",
        "supplierPower": "Assess the bargaining power of suppliers.",
        "competitiveRivalry": "Assess the intensity of competition among existing players.",
        "threatOfSubstitutes": "Assess the threat of substitute products or services.",
        "buyerPower": "Assess the bargaining power of buyers."
      },
      "keySuccessFactors": "Identify the critical competencies required to succeed in this industry (e.g., innovation, cost efficiency, distribution network)."
    },
    "analysisObjectivesAndScope": {
      "objectives": "Clearly define the goals of the analysis (e.g., assess product feature gaps, understand market share, identify new opportunities).",
      "scope": {
        "competitorTypes": "Clarify whether the analysis includes direct, indirect, or potential future competitors.",
        "geographicMarket": "Specify the geographical boundaries of the analysis."
      }
    },
    "internalSWOTAnalysis": {
      "opportunities": "Identify external opportunities revealed by the market analysis (e.g., underserved segments, market trends).",
      "weaknesses": "Acknowledge internal weaknesses and capability gaps where competitors are stronger.",
      "strengths": "List your company's internal strengths relative to competitors.",
      "threats": "List external threats from the competitive landscape (e.g., aggressive competitor moves, new entrants)."
    },
    "detailedCompetitorProfiles": [
      {
        "background": {
          "ownership": "Public or private.",
          "scale": "Company size, revenue, and number of employees.",
          "history": "Brief history and founding information."
        },
        "competitorName": "Competitor A",
        "targetMarket": "Describe the primary customer segments the competitor targets.",
        "pricingStrategy": "Outline the competitor's pricing model (e.g., premium, subscription, freemium).",
        "weaknesses": [
          "List key internal weaknesses (e.g., narrow product line, high prices)."
        ],
        "marketingAndPromotion": "Analyze the competitor's marketing channels, messaging, and brand positioning.",
        "distributionChannels": "Describe how the competitor delivers its products or services to the market.",
        "productsAndValueProposition": "Detail the competitor's key products, services, and unique selling points.",
        "strengths": [
          "List key internal strengths (e.g., brand recognition, technology)."
        ]
      }
    ],
    "executiveSummary": {
      "strategicRecommendations": "Summarize the primary actions and strategic changes suggested by the analysis.",
      "keyFindings": "Provide a high-level overview of the market, competitor landscape, and the company's position. Highlight the most critical insights."
    }
  }
  """

  # Set mock reviewer_guidelines from Firestore
  # These are the same regardless of creating a new doc or modifyin an existing one
  reviewer_guidelines = """
    ### \#\# Critique Instructions

    **1. Structural Validation:**

      * Validate the `updated_strategy_doc` against the BEST PRACTICES.
      * Identify any **missing** required keys.
      * Identify any keys with the **incorrect data type** (e.g., a `string` where an `array` is required).
      * Identify any **extra keys** present in the document that are not defined in the schema.

    **2. Content Completeness Check:**

      * For every field, verify that the provided content is **present and non-trivial**.
      * Flag any fields that are empty strings (`""`), `null`, or contain only placeholder text (e.g., "TBD", "[to be completed]", "Insert text here").
      * Flag any descriptions or analyses that are nonsensically short (e.g., a single word where a paragraph is expected), as this indicates incomplete generation.

    **3. Generate Actionable Feedback:**

      * For each error you uncover, provide a clear description of the issue and a specific suggestion for how to fix it.
      * Compile all findings into a single JSON object, following the **Output Format** specified below precisely. If there are no errors, the `errors` array should be empty and `isValid` should be `true`.

    -----

    ### \#\# Output Format (Strict)

    Your entire response must be a single JSON object matching this structure:

    ```json
    {
      "critiqueReport": {
        "isValid": false,
        "errors": [
          {
            "errorType": "STRUCTURAL_ERROR | CONTENT_ERROR",
            "fieldPath": "path.to.the.problematic.field",
            "issue": "A clear, concise description of what is wrong.",
            "suggestion": "A specific, actionable instruction on how to correct the error."
          }
        ]
      }
    }
    ```

    -----

    ### \#\# Example Critique

    **Example Issue:** The `valueProposition` key is missing from the `productsAndServices` object.

    **Your Corresponding Output Error Object:**

    ```json
    {
      "errorType": "STRUCTURAL_ERROR",
      "fieldPath": "productsAndServices.valueProposition",
      "issue": "The required key 'valueProposition' is missing from the object.",
      "suggestion": "Add the required key 'valueProposition' and provide a string value explaining the unique benefits the product solves for customers."
    }
    ```

    **Example Issue:** The `historyAndBackground` field contains only the word "History".

    **Your Corresponding Output Error Object:**

    ```json
    {
      "errorType": "CONTENT_ERROR",
      "fieldPath": "companyOverview.historyAndBackground",
      "issue": "The content for this field is incomplete and lacks detail.",
      "suggestion": "Expand this field to include details about the company's founding, major milestones, and its evolution over time."
    }
    ```
    """

## Run the Test

In [ ]:
# --- Test with existing strategy_doc
# TODO: Set the query parameter dynamically based on whether an existing doc has been created alreay or not
@weave.op()
async def run_sequential_app():
    #query = query
    worker_agent = worker_agents['iterative_strategy_agent']
    worker_session = await session_service.create_session(
        app_name=worker_agent.name,
        user_id=my_user_id,
    )

    await run_strategy_agent(
        agent=worker_agent,
        query=query, # Updated query to emphasize updating existing doc
        session=worker_session,
        user_id=my_user_id,
        strategy_doc=strategy_doc,  # This should have the existing document
        best_practices=best_practices,
        reviewer_guidelines=reviewer_guidelines,
        new_information=new_information  # This has the founding date
    )
    print(f"--- {worker_agent.name} Complete ---")

# Test the debugging
await run_sequential_app()

 (subsequent messages of this type will be suppressed)



🚀 Running query for agent: 'iterative_strategy_agent' in session: 'fed77afc-837c-4b45-8f93-75cb661c07fb'...
🔍 DEBUG: strategy_doc type: <class 'NoneType'>
🔍 DEBUG: strategy_doc is None: True
🔍 DEBUG: strategy_doc is empty or None
🔧 Agent type: SequentialAgent (composite agent)
📝 Creating new document
📋 Provided best_practices
✅ Provided reviewer_guidelines
📊 Provided new_information



📊 Token Usage Summary:
   • Total tool calls: 0

   Per-Agent Breakdown:
   • strategist_agent (gemini-1.5-pro-002):
     - Prompt: 3,222 tokens
     - Response: 3,085 tokens
     - Total: 6,307 tokens
   • reviewer_agent (gemini-1.5-flash-latest):
     - Prompt: 28,040 tokens
     - Response: 1,606 tokens
     - Total: 29,646 tokens
   • editor_agent (gemini-1.5-flash-latest):
     - Prompt: 30,507 tokens
     - Response: 9,257 tokens
     - Total: 39,764 tokens

   Overall Total:
   • Prompt tokens: 61,769
   • Response tokens: 13,948
   • Total tokens: 75,717

✅ FINAL RESPONSE:


```json
{
  "strategicRecommendations": {
    "monitoringPlan": "Continuously monitor competitor websites, social media, and industry publications for new product launches, marketing campaigns, and pricing changes. Track market share trends and customer reviews to identify emerging threats and opportunities. Use Google Alerts to stay informed about relevant industry news and competitor activities. Conduct quarterly competitor analysis reports to assess changes in the competitive landscape.",
    "leveragingStrengths": "Capitalize on Intellipure's patented DFS technology and superior air purification capabilities by highlighting its effectiveness in removing ultrafine particles, allergens, and viruses. Leverage the brand's reputation for quality and performance to target health-conscious consumers and those seeking premium air purification solutions. Partner with healthcare professionals and influencers to build credibility and expand reach.",
    "productAndServiceStrategy": "Expand the product line to offer portable air purifiers for travel and smaller spaces. Develop smart home integration features to enhance convenience and control. Explore subscription-based filter replacement services to generate recurring revenue. Offer customized air purification solutions for specific needs, such as allergy sufferers or pet owners.",
    "marketPositioning": "Position Intellipure as the leading provider of premium air purification solutions for health-conscious consumers. Emphasize the brand's commitment to clean air and healthy living. Highlight the superior performance and effectiveness of its patented DFS technology. Target marketing efforts towards consumers concerned about indoor air quality, allergies, and respiratory health.",
    "addressingCompetitiveGaps": "Enhance marketing efforts to improve brand visibility and awareness. Expand distribution channels to reach a wider audience. Offer competitive pricing and promotions to attract price-sensitive customers. Develop educational content to highlight the benefits of advanced air purification technology.",
    "marketingAndSalesStrategy": "Implement a multi-channel marketing strategy that includes digital advertising, social media marketing, content marketing, and influencer partnerships. Target health and wellness publications and websites. Offer educational webinars and workshops to demonstrate the benefits of Intellipure's technology. Develop a strong online presence and optimize the website for conversions. Implement a customer loyalty program to encourage repeat purchases.",
    "prioritizedOpportunities": "Focus on expanding market share in the residential and commercial sectors. Target healthcare facilities, schools, and other organizations with high indoor air quality needs. Develop strategic partnerships with HVAC companies and other related businesses. Expand into international markets with a focus on regions with high levels of air pollution."
  },
  "competitiveLandscape": {
    "marketShareAndPositioning": [
      {
        "marketSharePercentage": "requires further research",
        "competitorName": "Austin Air",
        "positioning": "High-quality, durable air purifiers with a focus on chemical and gas filtration."
      },
      {
        "marketSharePercentage": "requires further research",
        "competitorName": "IQAir",
        "positioning": "Premium air purifiers with advanced filtration technology and a focus on medical-grade performance."
      },
      {
        "marketSharePercentage": "requires further research",
        "competitorName": "Molekule",
        "positioning": "Innovative air purifiers with PECO technology that destroys pollutants at a molecular level."
      },
      {
        "marketSharePercentage": "requires further research",
        "competitorName": "AirDoctor",
        "positioning": "High-performance air purifiers with a focus on removing ultrafine particles and allergens."
      }
    ],
    "competitorSegmentation": {
      "strategicGroups": "Premium brands: IQAir, Molekule, AirDoctor. Value brands: Honeywell, Blueair, Coway. Niche brands: Austin Air (focus on chemical filtration), Rabbit Air (focus on design and customization)."
    },
    "competitorIdentification": [
      {
        "type": "Direct",
        "name": "Austin Air"
      },
      {
        "type": "Direct",
        "name": "IQAir"
      },
      {
        "type": "Direct",
        "name": "Molekule"
      },
      {
        "type": "Direct",
        "name": "AirDoctor"
      },
      {
        "type": "Indirect",
        "name": "Honeywell"
      },
      {
        "type": "Indirect",
        "name": "Blueair"
      },
      {
        "type": "Indirect",
        "name": "Coway"
      },
      {
        "type": "Indirect",
        "name": "Rabbit Air"
      }
    ]
  },
  "comparativeAnalysis": {
    "comparisonMatrix": "requires further research",
    "brandPositioningMap": {
      "description": "Visualizes competitor positions based on price and quality.",
      "axisY": "Quality",
      "axisX": "Price",
      "competitorPositions": [
        {
          "y": "High",
          "name": "IQAir",
          "x": "High"
        },
        {
          "y": "High",
          "name": "Molekule",
          "x": "High"
        },
        {
          "y": "Medium",
          "name": "AirDoctor",
          "x": "Medium"
        },
        {
          "y": "High",
          "name": "Austin Air",
          "x": "Medium"
        },
        {
          "y": "Medium",
          "name": "Honeywell",
          "x": "Low"
        },
        {
          "y": "High",
          "name": "Intellipure",
          "x": "High"
        }
      ]
    },
    "featureAndCapabilityGaps": "requires further research"
  },
  "marketAndIndustryOverview": {
    "pestelAnalysis": {
      "technological": "Advancements in sensor technology, filter materials, and smart home integration are driving innovation in the air purifier market. The development of new air purification technologies, such as PECO, is creating new opportunities for differentiation.",
      "political": "Government regulations related to air quality standards and energy efficiency can impact the industry. Tax incentives and subsidies for energy-efficient appliances can influence consumer demand.",
      "social": "Growing awareness of the health risks associated with poor indoor air quality is driving demand for air purifiers. Increasing consumer preference for sustainable and eco-friendly products is influencing product development.",
      "environmental": "Rising levels of air pollution and climate change concerns are driving demand for air purifiers. Regulations related to the disposal of air purifier filters can impact the industry.",
      "economic": "Economic growth and rising disposable incomes can drive demand for premium air purifiers. Economic downturns can lead to decreased consumer spending on non-essential appliances.",
      "legal": "Product safety regulations and labeling requirements can impact the industry. Intellectual property laws protect patented technologies and designs."
    },
    "marketDefinition": {
      "growthRate": "The global air purifier market is expected to grow at a CAGR of 8.9% from 2023 to 2030. (Source: https://www.grandviewresearch.com/industry-analysis/air-purifier-market)",
      "size": "The global air purifier market size was valued at USD 11.86 billion in 2022. (Source: https://www.grandviewresearch.com/industry-analysis/air-purifier-market)",
      "keyTrends": "Increasing demand for smart air purifiers with IoT connectivity. Growing adoption of HEPA and activated carbon filters. Rising demand for portable and compact air purifiers. Increasing focus on energy efficiency and sustainability."
    },
    "portersFiveForces": {
      "threatOfNewEntrants": "Moderate. The air purifier market has relatively low barriers to entry, but establishing brand recognition and distribution networks can be challenging.",
      "supplierPower": "Moderate. The supply chain for air purifier components is relatively fragmented, giving manufacturers some bargaining power.",
      "competitiveRivalry": "High. The air purifier market is highly competitive, with numerous established players and new entrants vying for market share.",
      "threatOfSubstitutes": "Low. There are limited substitutes for air purifiers, particularly for consumers with specific air quality concerns.",
      "buyerPower": "Moderate. Consumers have a wide range of choices in terms of brands, features, and price points, giving them some bargaining power."
    },
    "keySuccessFactors": "Product innovation and differentiation. Effective marketing and branding. Strong distribution network. Competitive pricing. Excellent customer service."
  },
  "analysisObjectivesAndScope": {
    "objectives": "Assess Intellipure's competitive position in the air purifier market. Identify key competitors and their strengths and weaknesses. Analyze market trends and growth opportunities. Develop strategic recommendations for Intellipure to enhance its competitive advantage.",
    "scope": {
      "competitorTypes": "Direct and indirect competitors in the residential and commercial air purifier market.",
      "geographicMarket": "United States."
    }
  },
  "internalSWOTAnalysis": {
    "opportunities": "Growing demand for premium air purifiers. Increasing awareness of indoor air quality issues. Expansion into new market segments, such as healthcare and education. Development of new product features and technologies.",
    "weaknesses": "Limited brand awareness compared to established competitors. Higher price point compared to some value brands. Limited distribution network.",
    "strengths": "Patented DFS technology with superior air purification capabilities. Strong reputation for quality and performance. Focus on health-conscious consumers.",
    "threats": "Intense competition from established players and new entrants. Price pressure from value brands. Changing consumer preferences and technological advancements."
  },
  "detailedCompetitorProfiles": [
    {
      "background": {
        "ownership": "Private",
        "scale": "requires further research",
        "history": "requires further research"
      },
      "competitorName": "Austin Air",
      "targetMarket": "Residential and commercial customers seeking high-quality air purification solutions for chemical and gas filtration.",
      "pricingStrategy": "Premium pricing",
      "weaknesses": [
        "Limited product line",
        "Traditional design"
      ],
      "marketingAndPromotion": "Focus on online marketing and direct sales. Partnerships with healthcare professionals.",
      "distributionChannels": "Online store, authorized dealers",
      "productsAndValueProposition": "Durable air purifiers with long-lasting filters. Effective removal of chemicals, gases, and odors.",
      "strengths": [
        "High-quality construction",
        "Long filter life"
      ]
    },
    {
      "background": {
        "ownership": "Private",
        "scale": "requires further research",
        "history": "requires further research"
      },
      "competitorName": "IQAir",
      "targetMarket": "Residential and commercial customers seeking premium air purification solutions with medical-grade performance.",
      "pricingStrategy": "Premium pricing",
      "weaknesses": [
        "High price point",
        "Bulky design"
      ],
      "marketingAndPromotion": "Focus on online marketing, social media, and influencer partnerships. Educational content and webinars.",
      "distributionChannels": "Online store, authorized dealers",
      "productsAndValueProposition": "Advanced filtration technology for superior air purification. Medical-grade performance for allergy and asthma sufferers.",
      "strengths": [
        "Advanced filtration technology",
        "Medical-grade performance"
      ]
    },
    {
      "background": {
        "ownership": "Private",
        "scale": "requires further research",
        "history": "requires further research"
      },
      "competitorName": "Molekule",
      "targetMarket": "Residential customers seeking innovative air purification solutions with PECO technology.",
      "pricingStrategy": "Premium pricing",
      "weaknesses": [
        "High price point",
        "Limited filter life"
      ],
      "marketingAndPromotion": "Focus on online marketing, social media, and influencer partnerships. Modern and stylish design.",
      "distributionChannels": "Online store, retail partners",
      "productsAndValueProposition": "PECO technology that destroys pollutants at a molecular level. Sleek and modern design.",
      "strengths": [
        "Innovative PECO technology",
        "Stylish design"
      ]
    },
    {
      "background": {
        "ownership": "Private",
        "scale": "requires further research",
        "history": "requires further research"
      },
      "competitorName": "AirDoctor",
      "targetMarket": "Residential customers seeking high-performance air purifiers for removing ultrafine particles and allergens.",
      "pricingStrategy": "Mid-range to premium pricing",
      "weaknesses": [
        "Limited brand awareness",
        "Simpler design"
      ],
      "marketingAndPromotion": "Focus on online marketing and performance-based advertising. Educational content and customer reviews.",
      "distributionChannels": "Online store, retail partners",
      "productsAndValueProposition": "High-performance filtration for removing ultrafine particles and allergens. User-friendly features and controls.",
      "strengths": [
        "Effective filtration",
        "User-friendly design"
      ]
    }
  ],
  "executiveSummary": {
    "strategicRecommendations": "Intellipure should focus on leveraging its patented DFS technology and strong reputation for quality to target health-conscious consumers. The company should enhance its marketing efforts to improve brand visibility and expand its distribution network. Product line expansion and strategic partnerships can further strengthen its competitive position.",
    "keyFindings": "The air purifier market is growing rapidly, driven by increasing awareness of indoor air quality issues. Intellipure faces competition from established players and new entrants, but its patented DFS technology and focus on health-conscious consumers provide a competitive advantage. The company has opportunities to expand its market share by enhancing marketing efforts, expanding its distribution network, and developing new product offerings."
  }
}
```


--- iterative_strategy_agent Complete ---
